# Post-Fire Landslide Susceptibility Modeling — Nacimiento, Chile
### Variable 2: Slope

**Author:** Constanza Morales Gajardo
**Part of:** Landslide Susceptibility Portfolio Project (P5)

## 1. Introduction

This notebook derives the slope variable from the SRTM Digital Elevation Model (DEM) for the Nacimiento study area, as the second conditioning variable for the post-fire landslide susceptibility model.

- **Study area:** Nacimiento commune (GADM boundary, `NAME_3`)
- **Data source:** SRTM Digital Elevation Model (USGS/SRTMGL1_003, 30m resolution)
- **Tools:** Google Earth Engine Python API, geemap, geopandas

In [1]:
import ee
import geemap
import geopandas as gpd
import os

# Initialize Earth Engine
ee.Initialize(project='earth-engine-portfolio-498609')

print("Earth Engine initialized successfully")

Earth Engine initialized successfully


## 2. Study Area — Nacimiento Commune Boundary

In [2]:
# Load Chile administrative boundaries (GADM level 3)
gadm_path = "../../nacimiento-wildfire-2023/data/raw/gadm41_CHL.gpkg"

if not os.path.exists(gadm_path):
    print(f"File not found at: {gadm_path}")
else:
    chile_admin = gpd.read_file(gadm_path, layer='ADM_ADM_3')
    nacimiento = chile_admin[chile_admin['NAME_3'] == 'Nacimiento']
    print(f"Nacimiento boundary loaded: {len(nacimiento)} feature(s)")

# Convert to Earth Engine geometry
aoi = geemap.geopandas_to_ee(nacimiento)

Nacimiento boundary loaded: 1 feature(s)


## 3. SRTM DEM and Slope Calculation

The SRTM (Shuttle Radar Topography Mission) DEM provides elevation data at 30m resolution. Slope is calculated in degrees using Earth Engine's `ee.Terrain.slope()` function, which computes the maximum rate of elevation change between each pixel and its neighbors.

In [3]:
# Load SRTM DEM, clipped to study area
dem = ee.Image('USGS/SRTMGL1_003').clip(aoi)

# Calculate slope in degrees
slope = ee.Terrain.slope(dem).rename('slope')

print("DEM loaded and slope calculated successfully")

DEM loaded and slope calculated successfully


## 4. Slope Visualization

In [5]:
# Visualization parameters for slope
slope_vis = {
    'min': 0,
    'max': 45,
    'palette': ['1a9850', '91cf60', 'fee08b', 'fc8d59', 'd73027']
}

Map = geemap.Map()
Map.centerObject(aoi, 11)
Map.addLayer(slope, slope_vis, 'Slope (degrees)')
Map.addLayer(aoi, {'color': 'black'}, 'Nacimiento boundary', False)
Map

Map(center=[-37.48548152276885, -72.82353068056327], controls=(WidgetControl(options=['position', 'transparent…

## 5. Slope Reclassification

Slope is reclassified into 5 susceptibility classes, following common thresholds used in landslide susceptibility literature:

| Class | Slope range (°) | Susceptibility |
|-------|-----------------|-----------------|
| 1 | 0–5° | Very Low |
| 2 | 5–15° | Low |
| 3 | 15–25° | Moderate |
| 4 | 25–35° | High |
| 5 | >35° | Very High |

These thresholds reflect the general relationship between slope angle and shear stress on hillslopes: steeper slopes increase gravitational driving forces relative to soil/rock shear resistance, increasing landslide likelihood — up to a point where very steep slopes (>45°) often have thinner soil cover and can be less susceptible to shallow landslides specifically (though more prone to rockfalls). For this model we treat higher slope as monotonically higher susceptibility, consistent with most post-fire debris flow literature.

**Note:** these thresholds will be cross-checked against the literature sources used for weighting in Session 4, and adjusted if needed for consistency.

In [6]:
# Reclassify slope into 5 susceptibility classes
slope_reclass = ee.Image(0) \
    .where(slope.gte(0).And(slope.lt(5)), 1) \
    .where(slope.gte(5).And(slope.lt(15)), 2) \
    .where(slope.gte(15).And(slope.lt(25)), 3) \
    .where(slope.gte(25).And(slope.lt(35)), 4) \
    .where(slope.gte(35), 5) \
    .rename('slope_class') \
    .clip(aoi)

print("Slope reclassified into 5 susceptibility classes")

Slope reclassified into 5 susceptibility classes


## 6. Reclassified Slope Visualization

In [7]:
# Visualization for reclassified slope (5 discrete classes)
reclass_vis = {
    'min': 1,
    'max': 5,
    'palette': ['1a9850', '91cf60', 'fee08b', 'fc8d59', 'd73027']
}

Map2 = geemap.Map()
Map2.centerObject(aoi, 11)
Map2.addLayer(slope_reclass, reclass_vis, 'Slope Susceptibility Class')
Map2.addLayer(aoi, {'color': 'black'}, 'Nacimiento boundary', False)
Map2

Map(center=[-37.485481522768936, -72.82353068056312], controls=(WidgetControl(options=['position', 'transparen…

## 7. Export Slope Layers as Static GeoTIFF

Exporting both the continuous slope (degrees) and the reclassified slope (5 susceptibility classes) as static rasters to `data/raw/`, ready to be used in the weighted overlay model (Session 4).

In [8]:
# Export continuous slope
slope_output_path = '../data/raw/nacimiento_slope.tif'
geemap.ee_export_image(
    slope,
    filename=slope_output_path,
    scale=30,
    region=aoi.geometry(),
    file_per_band=False
)
print(f"Slope (degrees) exported to: {slope_output_path}")

# Export reclassified slope
slope_reclass_output_path = '../data/raw/nacimiento_slope_reclass.tif'
geemap.ee_export_image(
    slope_reclass,
    filename=slope_reclass_output_path,
    scale=30,
    region=aoi.geometry(),
    file_per_band=False
)
print(f"Reclassified slope exported to: {slope_reclass_output_path}")

Generating URL ...
Please wait ...
Data downloaded to C:\Users\const\Documents\gis_projects\nacimiento-landslide-susceptibility\data\raw\nacimiento_slope.tif
Slope (degrees) exported to: ../data/raw/nacimiento_slope.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\const\Documents\gis_projects\nacimiento-landslide-susceptibility\data\raw\nacimiento_slope_reclass.tif
Reclassified slope exported to: ../data/raw/nacimiento_slope_reclass.tif


## Conclusions

- Slope successfully derived from SRTM DEM (30m resolution) for the Nacimiento study area.
- Slope reclassified into 5 susceptibility classes (Very Low to Very High), following standard landslide susceptibility thresholds.
- Both continuous slope and reclassified slope exported as static GeoTIFFs to `data/raw/`, ready for use in the weighted overlay model (Session 4).
- **Note:** slope thresholds will be cross-checked against literature sources gathered in Session 4 for consistency with assigned weights.
- **Next step (Session 3):** derive geology and land use/land cover variables.